# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ellarh101/ML-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**1. What one row means:** One row is one piece of content (one article or page) in March 2026. The data comes in daily rows, so I add up each item's daily numbers to get a single row for the month.

**2. Tables I'll use:** `dim_content`, which describes what each piece of content looks like (word count, category count, search volume, and so on), and `fact_content_daily_performance` for the month `2026-03`, which tells me how each piece performed.

**3. Time window:** 1 to 31 March 2026. I only keep content that already existed by 31 March. June 2026 stays sealed as my test month, and I won't use it to build anything.

**4. What I predict or rank:** Clustering has no true label, so I use a stand-in (a proxy): total March impressions per content item. I group the content by its structure, then check whether the groups differ in impressions. If they do, the groups mean something.

**5. What I deliberately leave out:** `content_updated_date`, `last_optimized_date` and `is_deleted`. These were recorded when the data was exported in July, not in March, so using them would let information from the future leak into a March picture.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from google.colab import userdata
from huggingface_hub import HfApi

token = userdata.get("HF_TOKEN")
files = HfApi().list_repo_files("FlyRank/internship-warehouse", repo_type="dataset", token=token)
for f in files:
    print(f)

.gitattributes
README.md
dim_clients.parquet
dim_content.parquet
fact_content_daily_performance/month=2025-01/data_0.parquet
fact_content_daily_performance/month=2025-02/data_0.parquet
fact_content_daily_performance/month=2025-03/data_0.parquet
fact_content_daily_performance/month=2025-04/data_0.parquet
fact_content_daily_performance/month=2025-05/data_0.parquet
fact_content_daily_performance/month=2025-06/data_0.parquet
fact_content_daily_performance/month=2025-07/data_0.parquet
fact_content_daily_performance/month=2025-08/data_0.parquet
fact_content_daily_performance/month=2025-09/data_0.parquet
fact_content_daily_performance/month=2025-10/data_0.parquet
fact_content_daily_performance/month=2025-11/data_0.parquet
fact_content_daily_performance/month=2025-12/data_0.parquet
fact_content_daily_performance/month=2026-01/data_0.parquet
fact_content_daily_performance/month=2026-02/data_0.parquet
fact_content_daily_performance/month=2026-03/data_0.parquet
fact_content_daily_performance/mont

In [2]:
import duckdb
from huggingface_hub import hf_hub_download

REPO = "FlyRank/internship-warehouse"

march_path = hf_hub_download(
    REPO, "fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset", token=token)
content_path = hf_hub_download(
    REPO, "dim_content.parquet", repo_type="dataset", token=token)
clients_path = hf_hub_download(
    REPO, "dim_clients.parquet", repo_type="dataset", token=token)

con = duckdb.connect()
for name, path in [("march daily fact", march_path),
                   ("dim_content", content_path),
                   ("dim_clients", clients_path)]:
    print("=====", name)
    print(con.sql(f"DESCRIBE SELECT * FROM '{path}'").df()[["column_name", "column_type"]])

===== march daily fact
                 column_name column_type
0                report_date        DATE
1             client_hash_id     VARCHAR
2            content_hash_id     VARCHAR
3             client_has_gsc     BOOLEAN
4             client_has_ga4     BOOLEAN
5         gsc_data_available     BOOLEAN
6         ga4_data_available     BOOLEAN
7            gsc_impressions      BIGINT
8                 gsc_clicks      BIGINT
9           gsc_sum_position      BIGINT
10          gsc_avg_position      DOUBLE
11             ga4_pageviews      BIGINT
12              ga4_sessions      BIGINT
13                 ga4_users      BIGINT
14      ga4_engaged_sessions      BIGINT
15  ga4_total_engagement_sec      BIGINT
16          sessions_organic      BIGINT
17           sessions_direct      BIGINT
18         sessions_referral      BIGINT
19           sessions_social      BIGINT
20             sessions_paid      BIGINT
21               sessions_ai      BIGINT
22                ai_chatgpt      

In [3]:
print(con.sql(f"SELECT * FROM '{march_path}' LIMIT 5").df().T)
print(con.sql(f"SELECT * FROM '{content_path}' LIMIT 5").df().T)

                                                 0                         1  \
report_date                    2026-03-01 00:00:00       2026-03-01 00:00:00   
client_hash_id             client_73cda7b4e4f265ea   client_73cda7b4e4f265ea   
content_hash_id           content_b7e512995f79d5a6  content_05597932fe4da067   
client_has_gsc                                True                      True   
client_has_ga4                               False                     False   
gsc_data_available                            True                      True   
ga4_data_available                            <NA>                      <NA>   
gsc_impressions                                 20                         1   
gsc_clicks                                       0                         0   
gsc_sum_position                                67                         0   
gsc_avg_position                              3.35                       0.0   
ga4_pageviews                           

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Feature** (describes the content itself, known by March 2026)
- `word_count`, `char_count`: how long the article is
- `category_count`: how many categories it belongs to
- `keyword_token_count`, `keyword_char_count`: how long the target keyword is
- `url_char_count`: how long the URL is
- `main_intent`, `content_type`: what kind of search intent and content type it is

**Label (my stand-in, not a true label)**
- Total March 2026 `gsc_impressions` per content item. I don't cluster on it. I use it afterwards to check whether the clusters mean something.

**Context** (used to find, join or filter rows, never as features)
- `content_hash_id`, `client_hash_id`, `keyword_hash_id`, `url_hash_id`: identifiers
- `report_date`, `month`: tell me which days belong to March
- `content_created_date`: used to keep only content that already existed by 31 March

**Excluded (with why)**
- `content_updated_date`, `last_optimized_date`, `optimization_eligible_date`, `is_deleted`, `is_published`: stamped at the July export, so they describe July, not March.
- `search_volume`, `competition`, `cpc`, `backlinks`: also come from the July snapshot, so I can't prove they had these values in March.
- `gsc_clicks`, `gsc_sum_position`, `gsc_avg_position`: same-month performance numbers that move together with impressions, so they would leak my proxy into the clusters.
- All `ga4_*`, `sessions_*`, `ai_*` and `scroll_events` columns: empty (missing) for the rows I looked at, so there is nothing to learn from.
- `provider_used`, `model_used`: the same value in every row I saw, so they can't separate anything.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
q1 = f"""
SELECT
    COUNT(*)                                                    AS total_rows,
    COUNT(DISTINCT (report_date, client_hash_id, content_hash_id)) AS distinct_day_client_content,
    COUNT(DISTINCT content_hash_id)                             AS distinct_content_items
FROM '{march_path}'
"""
print(con.sql(q1).df().T)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

                                   0
total_rows                   9841378
distinct_day_client_content  9841378
distinct_content_items        331437


**Query 1 finding:** The number of rows equals the number of distinct day + client + content combinations (9,841,378 each), so there are no duplicates and the daily grain is confirmed. March 2026 contains 331,437 distinct content items, which is how many rows my one-row-per-item table will have after I add up the days.

In [6]:
q2 = f"""
SELECT
    COUNT(*)                       AS row_count,
    MIN(report_date)               AS first_date,
    MAX(report_date)               AS last_date,
    COUNT(DISTINCT report_date)    AS distinct_days,
    COUNT(DISTINCT client_hash_id) AS distinct_clients
FROM '{march_path}'
"""
print(con.sql(q2).df().T)

                                    0
row_count                     9841378
first_date        2026-03-01 00:00:00
last_date         2026-03-31 00:00:00
distinct_days                      31
distinct_clients                   55


**Query 2 finding:** My March slice has 9,841,378 daily rows covering 1 to 31 March 2026, with all 31 days present. Only 55 of the 104 clients have data in this month, so the slice covers part of the panel, not all of it.

In [7]:
q3 = f"""
SELECT
    COUNT(*)                                                        AS total_rows,
    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE)              AS rows_surviving_is_true,
    COUNT(*) FILTER (WHERE gsc_data_available IS NOT TRUE)          AS rows_dropped,
    ROUND(100.0 * COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) / COUNT(*), 2) AS pct_surviving
FROM '{march_path}'
"""
print(con.sql(q3).df().T)

                                 0
total_rows              9841378.00
rows_surviving_is_true  3611061.00
rows_dropped            6230317.00
pct_surviving                36.69


**Query 3 finding:** Filtering with `gsc_data_available IS TRUE` keeps 3,611,061 of 9,841,378 March rows (36.69%). The other 6,230,317 rows are dropped because the flag is false or missing, so my analysis only sees the content that has real Search Console data.

In [8]:
frame_q = f"""
WITH march AS (
    SELECT content_hash_id,
           SUM(gsc_impressions) AS march_impressions
    FROM '{march_path}'
    WHERE gsc_data_available IS TRUE
    GROUP BY content_hash_id
)
SELECT c.content_hash_id,
       c.word_count,
       c.category_count,
       c.keyword_token_count,
       c.url_char_count,
       c.main_intent,
       m.march_impressions
FROM '{content_path}' c
JOIN march m USING (content_hash_id)
WHERE c.content_created_date <= DATE '2026-03-31'
"""
df = con.sql(frame_q).df()
print(df.shape)
print("unique content ids:", df["content_hash_id"].is_unique)
print(df.isna().sum())
df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(176738, 7)
unique content ids: True
content_hash_id            0
word_count             55315
category_count             0
keyword_token_count        0
url_char_count             0
main_intent            15839
march_impressions          0
dtype: int64


,content_hash_id,word_count,category_count,keyword_token_count,url_char_count,main_intent,march_impressions
0,content_7a105f548d9c6916,2123,0,5,100,informational,6523.0
1,content_a3ea9792f793ec72,<NA>,0,6,103,informational,453.0
2,content_36c36abc7650d7af,2546,0,7,130,informational,5630.0
3,content_a7da352b73b02668,2330,0,6,127,transactional,4944.0
4,content_1855a661b4d36130,<NA>,0,7,98,informational,429.0


**Decision moment:** the start of March 2026, before any March performance is observed.

Each feature is knowable at that moment because:

- `word_count`: the article's length is fixed when it is written, which is before it can earn any March impressions. (Limit: the table is a July snapshot, so an article edited after March could have a different length. Missing for about 31% of items.)
- `category_count`: categories are assigned when the content is created, so they are known before March.
- `keyword_token_count`: the target keyword is chosen before the article is written, so its length is known up front.
- `url_char_count`: the URL is set when the page is published, so its length is known before March traffic.
- `main_intent`: the search intent is a label of the target keyword, decided when the keyword is picked. (Missing for about 9% of items.)

**Missing values:** I leave gaps in `word_count` as missing (the model handles them) and treat missing `main_intent` as its own "unknown" group, so I don't drop a third of my data.

In [9]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

df["high_impressions"] = (df["march_impressions"] > df["march_impressions"].median()).astype(int)
print("share labelled high:", round(df["high_impressions"].mean(), 3))

features = ["word_count", "category_count", "keyword_token_count", "url_char_count", "main_intent"]

def quick_score(data, cols):
    X = pd.get_dummies(data[cols], columns=[c for c in cols if c == "main_intent"], dummy_na=True)
    y = data["high_impressions"]
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
    model = HistGradientBoostingClassifier(random_state=42).fit(X_tr, y_tr)
    return roc_auc_score(y_te, model.predict_proba(X_te)[:, 1])

honest_auc = quick_score(df, features)
print("honest AUC:", round(honest_auc, 3))

share labelled high: 0.5
honest AUC: 0.778


**The trap:** I added `leak_impressions_per_day`, which is March impressions divided by 31. That is just the label in a different form. The score jumped from 0.774 to 1.0, a perfect score, which is fake: the model was handed the answer. At the decision moment (the start of March) this number does not exist yet, so it can never be used in practice. I deleted the column and kept the honest score of 0.774.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**Limitation:** My slice covers only part of the panel. Query 2 showed that only 55 of the 104 clients have data in March, and Query 3 showed that only 36.69% of March rows have Search Console data available. The results describe the content of those clients with real Search Console data and may not hold for the others. The history is also unbalanced across clients, so a few large clients may dominate the picture. Finally, `dim_content` is a July snapshot, so a feature like word count may differ slightly from what it was in March.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.